# ⭕ Differentiable NMR (RDCs & NOEs)

Deep dive into the differentiable implementation of Saupe tensor fitting for Residual Dipolar Couplings (RDCs) and flat-bottomed harmonic penalties for NOE distance restraints.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/resonance-flow/blob/main/examples/interactive_tutorials/differentiable_nmr.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/elkins/resonance-flow.git git+https://github.com/elkins/diff-biophys.git biotite matplotlib
else:
    sys.path.append("../../")

import matplotlib.pyplot as plt
import numpy as np

## 1. Residual Dipolar Couplings (RDCs) & The Saupe Tensor

RDCs measure the angle between a bond vector (like N-H) and the external magnetic field. Because proteins tumble partially aligned in solution, these vectors trace out an alignment tensor (the Saupe tensor).

The equation relating the RDC $D_{ij}$ to the internuclear vector $\vec{v}_{ij}$ and the Saupe tensor $\mathbf{S}$ is:
$$ D_{ij} = D_{max} \cdot \vec{v}_{ij}^T \mathbf{S} \vec{v}_{ij} $$

ResonanceFlow computes RDCs differentially, meaning we can backpropagate through the Saupe tensor fitting process!

In [ ]:
import jax.numpy as jnp

from resonance_flow import rdc_loss, rdc_q_factor

np.random.seed(42)
n_vecs = 30
phi = np.random.uniform(0, np.pi * 2, n_vecs)
theta = np.random.uniform(0, np.pi / 2, n_vecs)
x = np.sin(theta) * np.cos(phi)
y = np.sin(theta) * np.sin(phi)
z = np.cos(theta)
nh_vecs = jnp.array(np.column_stack([x, y, z]))

S_true = jnp.array([[1.5, 0.2, -0.1], [0.2, -0.5, 0.3], [-0.1, 0.3, -1.0]]) * 10.0

measured_rdc = jnp.einsum("ni,ij,nj->n", nh_vecs, S_true, nh_vecs)
measured_rdc += np.random.normal(0, 1.0, n_vecs)

loss = rdc_loss(nh_vecs, measured_rdc)
print(f"RDC Loss: {loss:.3f}")

q = rdc_q_factor(nh_vecs, measured_rdc)
print(f"Q-factor: {q:.3f}")

## 2. Visualizing the NH Vectors and Alignment

We can plot the NH bond vectors in 3D space, color-coded by the magnitude and sign of their measured RDCs.

In [ ]:
import matplotlib.cm as cm

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

rdc_norm = plt.Normalize(np.min(measured_rdc), np.max(measured_rdc))
colors = cm.coolwarm(rdc_norm(measured_rdc))

for i in range(n_vecs):
    ax.quiver(
        0,
        0,
        0,
        nh_vecs[i, 0],
        nh_vecs[i, 1],
        nh_vecs[i, 2],
        color=colors[i],
        arrow_length_ratio=0.1,
        linewidth=2,
    )

ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("NH Bond Vectors Colored by Measured RDC")

sm = plt.cm.ScalarMappable(cmap=cm.coolwarm, norm=rdc_norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Measured RDC (Hz)", shrink=0.5)
plt.show()

## 3. Correlation: Measured vs. Calculated RDCs

In ResonanceFlow, `rdc_loss` uses Singular Value Decomposition (SVD) to find the absolute best-fit Saupe tensor for the given coordinates in a single differentiable step. Let's extract the calculated RDCs to see how well they fit the noisy measurements.

In [ ]:
from resonance_flow.losses import calculate_rdcs, fit_saupe_tensor

# Fit the Saupe tensor differentially
fitted_s = fit_saupe_tensor(nh_vecs, measured_rdc, d_max=1.0)

# Back-calculate the RDCs using the fitted tensor
calc_rdc = calculate_rdcs(nh_vecs, fitted_s, d_max=1.0)

plt.figure(figsize=(6, 6))
plt.scatter(measured_rdc, calc_rdc, color="purple", alpha=0.7, edgecolor="k", s=60)

min_val = min(float(np.min(measured_rdc)), float(np.min(calc_rdc))) - 2
max_val = max(float(np.max(measured_rdc)), float(np.max(calc_rdc))) + 2
plt.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.5, label="Ideal Fit")

plt.xlabel("Measured RDC (Hz)")
plt.ylabel("Calculated RDC (Hz)")
plt.title(f"RDC Correlation (Q-factor: {q:.3f})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. NOE Distance Restraints

Nuclear Overhauser Effect (NOE) spectroscopy gives us bounds on the distance between protons in space. In structural refinement, these are typically implemented as flat-bottomed harmonic potentials.

If the distance $r$ is within the bounds, the penalty is zero. If $r > U$, a harmonic penalty is applied. Let's visualize this penalty function!

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from resonance_flow.losses import noe_upper_bound_loss

distances = jnp.linspace(1.0, 8.0, 200)
upper_bound = 5.0


def eval_noe(d):
    positions = jnp.array([[0.0, 0.0, 0.0], [d, 0.0, 0.0]])
    noe_pairs = jnp.array([[0, 1]])
    upper_bounds = jnp.array([upper_bound])
    # The actual implementation doesn't use margin parameter if it's not in the signature
    # I will just pass positions, pairs, bounds
    return noe_upper_bound_loss(positions, noe_pairs, upper_bounds)


loss_values = jax.vmap(eval_noe)(distances)

plt.figure(figsize=(8, 4))
plt.plot(np.array(distances), np.array(loss_values), color="green", linewidth=3)
plt.axvline(upper_bound, color="red", linestyle="--", label=f"Upper Bound ({upper_bound} Å)")

plt.xlabel("Distance $r$ (Å)")
plt.ylabel("NOE Penalty Loss")
plt.title("Flat-Bottomed Harmonic Potential for NOE Restraints")
plt.legend()
plt.tight_layout()
plt.show()